In [0]:
import dlt

In [0]:
schema = spark.conf.get("spark.dataset.schema" , "labuser14322013_1774679369")
__order_status = spark.conf.get("order.status" , "NA")

In [0]:
schema_lcation= f"/Volumes/dbacademy/{schema}/landing/autoloader/schemas/1/"
dataset_load_path=f"/Volumes/dbacademy/{schema}/landing/files/"

#Define some rules for DQ

In [0]:
# Rules for DQ

__order_rules ={
    "valid_order_status" : "o_orderstatus in ('O', 'F' , 'P')",
    "valid_order_date" : "o_orderdate <= current_date()",
    "valid_order_price" : "o_totalprice >= 0"
}

__customer_rules = {
    "valid_customer_status" : "c_acctbal >= 0",
    "valid_market_segment" : "c_mktsegment in ('AUTOMOBILE', 'FURNITURE', 'HOUSEHOLD', 'MACHINERY')"
}

In [0]:
@dlt.table(
    table_properties={"layer" : "bronze"},
    comment="This is the raw order table read"
)
#@dlt.expect_all(__order_rules) # warn is default , all here applies ruleo on all records
#@dlt.expect_all_or_fail(__order_rules)
#@dlt.expect_all_or_drop(__order_rules)
def orders_stream_bronze():
  df= spark.readStream.table(f"dbacademy.{schema}.raw_order")
  return df

In [0]:
@dlt.table(
    table_properties={"layer" : "bronze"},
    comment="This is the raw customer table read",
    name="customer_bronze"
)
#@dlt.expect_all_or_fail(__customer_rules)
#@dlt.expect_all_or_drop(__customer_rules)
@dlt.expect_all(__customer_rules)  # warn default mode
def cust_bronze():
  df= spark.table(f"dbacademy.{schema}.raw_customer")
  return df

In [0]:
@dlt.table(
    table_properties={"layer" : "bronze"},
    comment="This is the raw order table ingesting data using Autoloader",
    name = "orders_autoloader_bronze"
)
def auto_orders_bronze():
   
    df= (spark.readStream.format("cloudFiles")
      .option("cloudFiles.format", "CSV")
      .option("cloudFiles.schemaLocation",schema_lcation)
      .option("cloudFiles.schemaHints", "o_orderkey long, o_custkey long, o_orderstatus string, o_totalprice decimal(18,2), o_orderdate date,     o_orderpriority string, o_clerk string, o_shippriority integer, o_comment string")
      .option("pathGlobFilter", "*.csv")
      .option("cloudFiles.schemaEvolutionMode", "none")
      .load(dataset_load_path))
    
   
    return df

In [0]:
# Append Flow is required o process data incrementally
# If we use SQL UNION everytime all the data will be processes 

dlt.create_streaming_table(
    name= "raw_order_union",
    comment="This is the raw order table loaded with union of autload and streamng raw order")


@dlt.append_flow(
    target="raw_order_union"
)
def order_streaming_table_to_target_append():
    df=spark.readStream.table("LIVE.orders_stream_bronze")
    return df


@dlt.append_flow(
    target="raw_order_union"
)
def order_autoloader_table_to_target_append():
    df=spark.readStream.table("LIVE.orders_autoloader_bronze")
    return df


In [0]:
@dlt.view(

    comment="This is the raw view after joining order and customer tables",

)
@dlt.expect_all(__order_rules) # warn is default , all here applies ruleo on all records
#@dlt.expect_all_or_fail(__order_rules)
#@dlt.expect_all_or_drop(__order_rules)
def joined_view():
  df= spark.sql("""
                select * from LIVE.raw_order_union o
                left join LIVE.customer_bronze c
                on o.o_custkey = c.c_custkey
                """)
  return df

In [0]:
import pyspark.sql.functions as F

@dlt.table(
    table_properties={"layer" : "silver"},
    comment="This is the cleaned silver layer  order and customer joined data"
)
def orders_silver():
  df= spark.read.table("LIVE.joined_view").withColumn("update_date" , F.current_timestamp())
  return df

In [0]:
@dlt.table(
    table_properties={"layer" : "gold"},
    comment="This is the  gold layer  order and customer joined data"
)
def orders_gold():
  df= spark.read.table("LIVE.orders_silver")

  df_agg = df.groupBy("c_mktsegment").agg(F.count("o_orderkey").alias("total_orders") , F.sum("o_totalprice").alias("total_revenue")).withColumn("update_timstamp" , F.current_timestamp())
  return df_agg

In [0]:
if __order_status != "NA":
    for o_status in __order_status.split(","):
        
        @dlt.table(
        table_properties={"layer" : "gold"},
        comment="This is the  gold layer  order and customer joined data",
        name = f"order_agg_{o_status}_gold"
                 )
        def func(o_status=o_status):
        
            df= spark.read.table("LIVE.orders_silver")

            df_agg = df.filter(f"o_orderstatus='{o_status}'")

            df_final = df_agg.groupBy("c_mktsegment").agg(F.count("o_orderkey").alias("total_orders") , F.sum("o_totalprice").alias("total_revenue")).withColumn("update_timstamp" , F.current_timestamp())
            
            return df_final